# Colab LLM Station

適用於 Google Colab 與雲端 GPU 環境之模組化推論與網路通道部署架構。
完整支援雙推論引擎（**vLLM** / **Ollama**）與雙網路穿透模式（**Tailscale 私有網格** / **Cloudflare 臨時公開通道**）。

---

## 步驟 0：拉取專案、注入環境變數與硬體檢測
直接拉取 GitHub 專案至 Colab 內部儲存空間（`/content/colab-llm-station`），檢測當前分配之 GPU 型號與顯存大小，並注入環境變數。

In [ ]:
# ==============================================================================
# 步驟 0: 拉取儲存庫、注入 Secrets 並檢測 GPU
# ==============================================================================
import os
from google.colab import userdata

# 1. 注入 Colab Secrets（若有設定）
try:
    os.environ['TAILSCALE_AUTHKEY'] = userdata.get('TAILSCALE_AUTHKEY')
except Exception:
    pass

try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

# 2. 自 GitHub 複製專案儲存庫（若已存在則自動 pull 更新）
!git clone https://github.com/yixuanw99/colab-llm-station.git /content/colab-llm-station 2>/dev/null || (cd /content/colab-llm-station && git pull)

# 3. 輸出硬體規格與推薦引擎
!cd /content/colab-llm-station && bash llm.sh status


## [選用] 遠端存取：Tailscale SSH 與本機 VS Code 直連設定

> **重要說明**：本步驟為**完全選用（OPTIONAL）**。
>
> 如果你只是想要讓模型在背景提供 HTTP API 端點（透過 Cloudflare 公開 HTTPS 或 Tailnet 私網）供 Python、OpenCode、Cursor 或外部呼叫，**不需要使用終端機修改檔案，請直接跳過此步驟**，進入「步驟 1」。

---

### 架構原理：為何需要 Tailscale Auth Key？
Google Colab 執行個體位於內部 NAT 網路後方，未配置獨立公網 IPv4 位址。使用具備暫態屬性（Ephemeral）的 Tailscale Auth Key，可在虛擬機器啟動時自動完成身分認證並建立網路節點，無需在執行過程中透過瀏覽器手動登入或掃描 QR Code。Tailscale SSH 透過 WireGuard 節點身分驗證，完全免去設定 SSH Key 與輸入密碼。

### 步驟 A：於 Tailscale Admin Console 產生金鑰
1. 開啟 [Tailscale Admin Console - Settings - Keys](https://login.tailscale.com/admin/settings/keys)。
2. 點擊 **Generate auth key**。
3. 設定金鑰各項參數：
   - **Description**: 輸入識別名稱（例如 `colab-llm-station`）。
   - **Reusable**: **開啟 (Enabled)**。允許同一金鑰在 Colab Runtime 重啟後重複使用，無需每次重新產生。
   - **Ephemeral**: **開啟 (Enabled，重要)**。當 Colab 執行個體中斷連線或關閉時，系統會自動自 Tailnet 裝置清單中移除該節點，避免累積無效離線主機。
   - **Pre-authorized**: **開啟 (Enabled)**。設備連線後自動核准加入網路，無需至後台手動審批。
   - **Tags**（選用）: 若組織 Tailnet ACL 規範需要標籤，可指定標籤（如 `tag:server`）。
4. 點擊 **Generate key** 並複製產生的金鑰字串（格式為 `tskey-auth-...`）。請妥善保存，Tailscale 僅會顯示此金鑰一次。

### 步驟 B：於 Google Colab Secrets 儲存金鑰與環境變數
1. 在 Google Colab 左側工具列點擊 **Secrets** 面板（鑰匙圖示）。
2. 點擊 **Add new secret**（新增密鑰）。
3. 填入設定數值：
   - **名稱 (Name)**：`TAILSCALE_AUTHKEY`（大小寫需完全一致）。
   - **值 (Value)**：貼上剛才複製的 `tskey-auth-...` 金鑰字串。
   - *(選用)* 若欲下載受權限保護之 Hugging Face 模型，可新增 `HF_TOKEN` 並填入存取權杖。
4. 將 **Notebook access**（筆記本存取權限）開關切換為 **開啟 (ON)**，以允許 Python 透過 `google.colab.userdata` 讀取金鑰。

### 步驟 C：本機電腦連線方式（執行下方儲存格後）

#### 方式 1：本機終端機直連（PowerShell / macOS Terminal / Linux）
```bash
ssh root@colab-llm-station
```
*(Tailscale 會自動進行身分驗證，完全免輸入密碼、免設定金鑰！)*

#### 方式 2：本機 VS Code (Remote - SSH) 遠端開發
1. 在本機 VS Code 安裝官方擴充套件 **「Remote - SSH」** (`ms-vscode-remote.remote-ssh`)。
2. 按 `F1` 鍵，輸入並選擇 `Remote-SSH: Connect to Host...`。
3. 輸入 `root@colab-llm-station`。
4. 連入後點擊「開啟資料夾」，選取 `/content/colab-llm-station` 即可開始開發！
*(亦相容於 Cursor、Zed、PyCharm Gateway 等工具)*。

In [ ]:
# ==============================================================================
# [選用] 獨立 Tailscale SSH 連線儲存格
# 僅在需要免密碼 SSH 連線或本機 VS Code (Remote - SSH) 遠端開發時執行
# ==============================================================================
authkey = os.environ.get('TAILSCALE_AUTHKEY') or userdata.get('TAILSCALE_AUTHKEY')
!cd /content/colab-llm-station && bash llm.sh tunnel tailscale up "$authkey"

print("\n" + "="*60)
print("[OK] Tailscale SSH 連線已就緒。")
print("1. 本機終端機連線指令：")
print("   ssh root@colab-llm-station")
print("\n2. 本機 VS Code (Remote - SSH) 連線：")
print("   按 F1 -> 輸入 Remote-SSH: Connect to Host... -> 輸入 root@colab-llm-station")
print("   （工作區目錄：/content/colab-llm-station）")
print("="*60)


## 步驟 1: 環境依賴初始化（依目標推論引擎安裝）

請根據步驟 0 檢測出之 GPU 規格選取對應安裝模式：
- **僅安裝 Ollama** (`bash setup.sh ollama`)：極速安裝（約 15 秒），推薦 **T4 / L4** 標準級 GPU。
- **僅安裝 vLLM** (`bash setup.sh vllm`)：安裝完整 PagedAttention 套件（約 2-3 分鐘），推薦 **A100 / H100** 企業級 GPU。
- **全安裝** (`bash setup.sh`)：同時安裝 Ollama 與 vLLM 所有相依套件。

In [ ]:
%%bash
cd /content/colab-llm-station

# 選項 A（標準級 T4/L4）：推薦極速安裝 Ollama
bash setup.sh ollama

# 選項 B（企業級 A100/H100）：安裝 vLLM
# bash setup.sh vllm

# 選項 C：雙引擎完整安裝
# bash setup.sh


## 硬體規格、資源佔用與推薦模型矩陣

請參考下表選取最適合你當前環境的組合（包含磁碟儲存空間、運行顯存與上下文長度）：

| GPU 等級 | 可用顯存 (VRAM) | 推薦引擎 | 推薦模型 | 量化格式 | 模型磁碟大小 | 運行佔用顯存 | 上下文長度 | 適用情境 |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **NVIDIA T4** | 15 GB GDDR6 | **Ollama** | `qwen2.5-coder:7b` | GGUF (Q4_K_M) | ~4.7 GB | ~6.5 GB | 32,768 | 免費版 Colab 輕量寫程式與日常對話 |
| **NVIDIA T4** | 15 GB GDDR6 | **Ollama** | `deepseek-r1:8b` | GGUF (Q4_K_M) | ~4.9 GB | ~7.2 GB | 32,768 | 思維鏈邏輯與數學推理 |
| **NVIDIA T4** | 15 GB GDDR6 | **Ollama** | `qwen2.5-coder:14b` | GGUF (Q4_K_M) | ~9.0 GB | ~12.5 GB | 16,384 | T4 15GB 顯存能運行的最高階寫程式模型 |
| **NVIDIA L4** | 24 GB GDDR6 | **Ollama** | `qwen2.5-coder:32b` | GGUF (Q4_K_M) | ~19.0 GB | ~20.5 GB | 32,768 | 本機 VS Code / OpenCode 主力寫程式模型 |
| **NVIDIA L4** | 24 GB GDDR6 | **Ollama** | `qwen3.8:27b` | GGUF (Q4_K_M) | ~17.0 GB | ~18.5 GB | 65,536 | 長文本對話、摘要與多輪整合 |
| **NVIDIA L4** | 24 GB GDDR6 | **Ollama** | `deepseek-r1:32b` | GGUF (Q4_K_M) | ~20.0 GB | ~21.5 GB | 32,768 | 高階演算法設計與架構推理 |
| **NVIDIA L4** | 24 GB GDDR6 | **vLLM** | `Qwen/Qwen2.5-Coder-14B-Instruct-AWQ` | 4-bit AWQ | ~9.5 GB | ~14.0 GB | 32,768 | 高併發推論服務 |
| **NVIDIA A100** | 40 / 80 GB HBM2 | **vLLM** | `Qwen/Qwen2.5-72B-Instruct-AWQ` | 4-bit AWQ | ~42.0 GB | ~45.0 GB | 32,768 | 旗艦級 72B 量化模型推論 |
| **NVIDIA A100** | 80 GB HBM2 | **vLLM** | `Qwen/Qwen2.5-Coder-32B-Instruct` | bfloat16 | ~65.0 GB | ~68.0 GB | 32,768 | 無損精度頂尖寫程式推論 |
| **NVIDIA A100** | 80 GB HBM2 | **vLLM** | `deepseek-ai/DeepSeek-R1-Distill-Qwen-32B` | bfloat16 | ~65.0 GB | ~68.0 GB | 32,768 | 無損精度高階思維鏈推理 |

---

## 部署方案 A: Ollama + Cloudflare Tunnel

### 方案特點
- **適用硬體**：NVIDIA T4 / L4
- **網路入口**：Cloudflare 臨時公開 HTTPS 通道 (`https://*.trycloudflare.com`)
- **必備金鑰**：無。完全免註冊、零配置即開即用。
- **最佳情境**：快速驗證 POC、向外部展示模型端點、或在未安裝 Tailscale 的電腦上呼叫 API。

In [ ]:
%%bash
cd /content/colab-llm-station

# 1. 下載目標模型（T4 推薦 qwen2.5-coder:7b，L4 可改為 qwen2.5-coder:32b）
MODEL="qwen2.5-coder:7b"
bash llm.sh ollama pull "$MODEL"

# 2. 啟動背景 Ollama 守護程序 (Port 11434)
bash llm.sh ollama serve

# 3. 建立 Cloudflare 公開入口通道並輸出 HTTPS 網址
bash llm.sh tunnel cloudflare 11434


## 部署方案 B: Ollama + Tailscale Mesh

### 方案特點
- **適用硬體**：NVIDIA T4 / L4
- **網路入口**：點對點加密私有網格 (`http://colab-llm-station:11434/v1`)
- **必備金鑰**：Colab Secrets 需設定 `TAILSCALE_AUTHKEY`。
- **最佳情境**：正式開發環境，直連本機 OpenCode、Cursor、Aider、LangChain 等工具，零公網暴露，安全穩定。

In [ ]:
%%bash
cd /content/colab-llm-station

# 1. 確保已連接至 Tailscale 網路
bash llm.sh tunnel tailscale up "$TAILSCALE_AUTHKEY"

# 2. 下載並啟動模型
MODEL="qwen2.5-coder:7b"  # L4 環境建議改為 qwen2.5-coder:32b
bash llm.sh ollama pull "$MODEL"
bash llm.sh ollama serve

# 3. 開放 Port 11434 至私有 Tailnet
bash llm.sh tunnel tailscale serve 11434

echo "================================================================"
echo "Tailnet 私網端點: http://colab-llm-station:11434/v1"
echo "================================================================"


## 部署方案 C: vLLM + Tailscale Mesh

### 方案特點
- **適用硬體**：NVIDIA A100 / H100 (或配備 AWQ 之 L4)
- **網路入口**：點對點加密私有網格 (`http://colab-llm-station:8000/v1`)
- **必備金鑰**：Colab Secrets 需設定 `TAILSCALE_AUTHKEY`。
- **核心優勢**：連續批處理 (Continuous Batching)、PagedAttention、最高併發吞吐量、標準 OpenAI API 相容。

In [ ]:
%%bash
cd /content/colab-llm-station

# 1. 確保已連接至 Tailscale 網路
bash llm.sh tunnel tailscale up "$TAILSCALE_AUTHKEY"

# 2. 開放 Port 8000 至私有 Tailnet
bash llm.sh tunnel tailscale serve 8000

# 3. 啟動 vLLM 推論引擎
# A100-40G 建議使用 Qwen/Qwen2.5-72B-Instruct-AWQ；A100-80G 可直跑 Qwen/Qwen2.5-Coder-32B-Instruct
MODEL="Qwen/Qwen2.5-Coder-32B-Instruct"
bash llm.sh vllm serve "$MODEL"

echo "================================================================"
echo "Tailnet 私網端點: http://colab-llm-station:8000/v1"
echo "================================================================"


## 部署方案 D: vLLM + Cloudflare Tunnel

### 方案特點
- **適用硬體**：NVIDIA A100 / H100
- **網路入口**：Cloudflare 臨時公開 HTTPS 通道 (`https://*.trycloudflare.com`)
- **必備金鑰**：無。零配置即開即用。
- **最佳情境**：在高效能 GPU 上進行 vLLM 推論基準評測，無需在本機安裝任何用戶端軟體。

In [ ]:
%%bash
cd /content/colab-llm-station

# 1. 啟動 vLLM 推論服務 (Port 8000)
MODEL="Qwen/Qwen2.5-72B-Instruct-AWQ"
bash llm.sh vllm serve "$MODEL"

# 2. 將 Port 8000 透過 Cloudflare 穿透至公網
bash llm.sh tunnel cloudflare 8000


## 步驟 3: 服務可用性診斷與狀態檢視

自動檢測當前運行之推論引擎通訊埠（8000 或 11434），並發送測試聊天請求以檢驗首字延遲 (TTFT) 與生成速度。

In [ ]:
# ==============================================================================
# 自動推論診斷與系統狀態檢測
# ==============================================================================
import json
import time
import urllib.request

# 1. 自動偵測活躍中之通訊埠（8000 為 vLLM，11434 為 Ollama）
port = None
for test_port in [8000, 11434]:
    try:
        req = urllib.request.Request(f"http://127.0.0.1:{test_port}/v1/models", headers={"User-Agent": "ColabDiagnostic"})
        with urllib.request.urlopen(req, timeout=2) as resp:
            if resp.status == 200:
                port = test_port
                data = json.loads(resp.read().decode('utf-8'))
                model_name = data['data'][0]['id']
                break
    except Exception:
        continue

if not port:
    print("[WARN] 未檢測到運行中之推論服務（Port 8000 或 11434 未回應）。")
    print("請先執行上方任一方案（方案 A、B、C 或 D）啟動推論引擎。")
else:
    print(f"[OK] 檢測到活躍服務於 Port {port} (載入模型: {model_name})")
    print("正在發送測試請求...")
    
    payload = {
        "model": model_name,
        "messages": [
            {"role": "system", "content": "你是一位頂尖的軟體架構工程師。"},
            {"role": "user", "content": "請簡要列出分散式系統中使用 PagedAttention 技術的三大核心優勢。"}
        ],
        "temperature": 0.2,
        "max_tokens": 256
    }
    
    t0 = time.time()
    req = urllib.request.Request(
        f"http://127.0.0.1:{port}/v1/chat/completions",
        data=json.dumps(payload).encode('utf-8'),
        headers={"Content-Type": "application/json", "Authorization": "Bearer colab"}
    )
    
    with urllib.request.urlopen(req) as resp:
        res = json.loads(resp.read().decode('utf-8'))
        duration = time.time() - t0
        content = res["choices"][0]["message"]["content"]
        usage = res.get("usage", {})
        
    print("\n" + "="*60)
    print(f"模型回答內容 ({model_name}):")
    print("="*60)
    print(content.strip())
    print("="*60)
    print(f"生成耗時: {duration:.2f} 秒 | 生成 Token 數: {usage.get('completion_tokens', 'N/A')}")

# 2. 檢視完整執行狀態
!cd /content/colab-llm-station && bash llm.sh status


## 步驟 4: 安全關閉與釋放 GPU 資源 (Teardown & Unassign)

使用完畢後，請務必依循以下程序終止執行階段。**僅關閉瀏覽器分頁不會停止 Colab VM 運作，仍會持續扣除 GPU 運算單元 (Compute Units)**。

### 釋放程序包含：
1. **停止背景服務**：終止推論引擎 (vLLM / Ollama) 與網路隧道 (Tailscale / Cloudflare)。
2. **登出 Tailscale**：清除本機節點註冊狀態（避免控制台殘留離線設備）。
3. **終止 Colab 執行階段**：呼叫 `runtime.unassign()` 立即釋放 GPU 實例並停止計費。

In [ ]:
# ==============================================================================
# 安全關閉服務並終止 Colab 執行階段（停止計費）
# ==============================================================================
import os
import subprocess

print('[INFO] 正在清理背景服務與網路通道...')
subprocess.run('bash /content/colab-llm-station/llm.sh vllm stop || true', shell=True)
subprocess.run('bash /content/colab-llm-station/llm.sh ollama stop || true', shell=True)
subprocess.run('tailscale serve reset || true', shell=True)
subprocess.run('tailscale logout || true', shell=True)
subprocess.run('pkill -f cloudflared || true', shell=True)

print('[INFO] 服務已終止。正在釋放 Colab 執行階段（停止 GPU 扣點）...')
try:
    from google.colab import runtime
    runtime.unassign()
except Exception as e:
    print(f'[NOTE] 無法透過程式碼自動中斷，請手動點擊 Colab 右上角『已連線』選單 -> 『中斷連線並刪除執行階段』。詳細訊息: {e}')
